In [9]:
%pip install azure-core
%pip install azure-search
%pip install azure-search-documents
%pip install azureml

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import json
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchFieldDataType,
    VectorSearch,
    VectorSearchAlgorithmConfiguration,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticField,
    SemanticSearch,
    SearchField
)
import openai
from tenacity import retry, wait_random_exponential, stop_after_attempt
from typing import List, Dict, Any


In [13]:
#https://learn.microsoft.com/en-us/azure/ai-foundry/ai-services/how-to/connect-azure-openai

# Configuration
AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_INDEX_NAME = "product-knowledge-base"
AZURE_SEARCH_KEY = os.environ["AZURE_SEARCH_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
EMBEDDING_MODEL_DEPLOYMENT = "text-embedding-ada-002"
VECTOR_DIMENSION = 1536  # Dimension for text-embedding-ada-002

KeyError: 'AZURE_SEARCH_SERVICE_ENDPOINT'

In [ ]:
search_index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, 
    credential=AzureKeyCredential(AZURE_SEARCH_KEY)
)

search_client = SearchClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX_NAME,
    credential=AzureKeyCredential(AZURE_SEARCH_KEY)
)

openai.api_type = "azure"
openai.api_version = "2023-12-01-preview"
openai.api_base = AZURE_OPENAI_ENDPOINT
openai.api_key = AZURE_OPENAI_KEY

In [ ]:
# Create the search index with vector search capabilities
def create_search_index() -> None:
    index = SearchIndex(
        name=AZURE_SEARCH_INDEX_NAME,
        fields=[
            SimpleField(name="id", type=SearchFieldDataType.String, key=True),
            SearchableField(name="title", type=SearchFieldDataType.String),
            SearchableField(name="content", type=SearchFieldDataType.String),
            SimpleField(name="category", type=SearchFieldDataType.String, filterable=True),
            SimpleField(name="product_id", type=SearchFieldDataType.String, filterable=True),
            SearchField(
                name="contentVector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True,
                vector_search_dimensions=VECTOR_DIMENSION,
                vector_search_profile_name="vectorConfig"
            )
        ],
        vector_search=VectorSearch(
            algorithms=[
                VectorSearchAlgorithmConfiguration(
                    name="hnsw",
                    kind="hnsw",
                    hnsw_parameters=HnswAlgorithmConfiguration(
                        m=4,
                        ef_construction=400,
                        ef_search=500,
                        metric="cosine"
                    )
                )
            ],
            profiles=[
                VectorSearchProfile(
                    name="vectorConfig",
                    algorithm_configuration_name="hnsw",
                )
            ]
        ),
        semantic_settings=SemanticSettings(
            configurations=[
                SemanticConfiguration(
                    name="my-semantic-config",
                    prioritized_fields=SemanticField(
                        content_fields=[{"field_name": "content"}],
                        title_fields=[{"field_name": "title"}]
                    )
                )
            ]
        )
    )
    
    search_index_client.create_or_update_index(index)
    print(f"Created search index: {AZURE_SEARCH_INDEX_NAME}")

In [ ]:
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def generate_embedding(text: str) -> List[float]:
    """Generate embedding for the given text using Azure OpenAI"""
    response = openai.Embedding.create(
        input=text,
        engine=EMBEDDING_MODEL_DEPLOYMENT
    )
    return response['data'][0]['embedding']

In [ ]:

def upload_documents(documents: List[Dict[str, Any]]) -> None:
    """Process and upload documents to Azure AI Search"""
    # Add vectors to documents
    for document in documents:
        # Generate embedding for content
        document["contentVector"] = generate_embedding(document["content"])
    
    # Upload in batches of 10
    batch_size = 10
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i + batch_size]
        results = search_client.upload_documents(documents=batch)
        print(f"Uploaded batch {i // batch_size + 1} - Success: {results.succeeded_count}, Failed: {results.failed_count}")

In [ ]:
def search_documents(query_text: str, top_k: int = 5, product_filter: str = None) -> List[Dict[str, Any]]:
    """Search documents using vector search with optional filtering"""
    # Generate embedding for query
    query_vector = generate_embedding(query_text)
    
    # Create filter if product_id is specified
    filter_expression = None
    if product_filter:
        filter_expression = f"product_id eq '{product_filter}'"
    
    # Perform vector search
    results = search_client.search(
        search_text=None,  # No traditional keyword search
        vector={"contentVector": query_vector, "k": top_k, "fields": "contentVector"},
        filter=filter_expression,
        select=["id", "title", "content", "category", "product_id"],
        top=top_k
    )
    
    # Convert results to list of documents
    docs = []
    for result in results:
        docs.append({
            "id": result["id"],
            "title": result["title"],
            "content": result["content"],
            "category": result["category"],
            "product_id": result["product_id"],
        })
    
    return docs

In [ ]:
def rag_response(query: str, product_id: str = None) -> str:
    """Generate a RAG response using Azure OpenAI and AI Search"""
    # Get relevant documents
    relevant_docs = search_documents(query, top_k=3, product_filter=product_id)
    
    # Format context from retrieved documents
    context = "\n\n".join([
        f"Title: {doc['title']}\nCategory: {doc['category']}\nContent: {doc['content']}"
        for doc in relevant_docs
    ])
    
    # Generate response using Azure OpenAI
    system_message = """
    You are a helpful customer support assistant for XYZ Electronics. 
    Your role is to assist customers with product inquiries, troubleshooting, 
    and return policies. Always be professional, empathetic, and solution-oriented.
    Use ONLY the information provided in the context to answer the question.
    If the answer is not in the context, politely say you don't have that specific information.
    """
    
    response = openai.ChatCompletion.create(
        engine="gpt-4-turbo",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": f"Context:\n{context}\n\nCustomer question: {query}"}
        ],
        temperature=0.3,
        max_tokens=500
    )
    
    return response.choices[0].message["content"]

In [ ]:
create_search_index()
    
# Example documents
sample_docs = [
    {
        "id": "doc1",
        "title": "XYZ Smart TV - Troubleshooting Guide",
        "content": "If your XYZ Smart TV won't turn on, first check if the power cable is securely connected. Then try unplugging the TV for 30 seconds and plugging it back in. If the issue persists, check if the remote has working batteries. For flickering screens, go to Settings > Display and adjust the refresh rate.",
        "category": "Troubleshooting",
        "product_id": "TV-2000X"
    },
    {
        "id": "doc2",
        "title": "XYZ Electronics Return Policy",
        "content": "All XYZ Electronics products can be returned within 30 days of purchase with the original receipt. Products must be in their original packaging with all accessories. Damaged items may not be eligible for a full refund. Special order items are non-refundable.",
        "category": "Policies",
        "product_id": "ALL"
    }
]

# Upload sample documents
upload_documents(sample_docs)

In [ ]:
customer_query = "My new TV screen keeps flickering, how can I fix it?"
response = rag_response(customer_query, product_id="TV-2000X")
print(f"\nCustomer: {customer_query}")
print(f"\nAssistant: {response}")